# Split VRM alarm logs by selected site

This notebook reads the cleaned VRM alarm log CSV, filters the same site IDs used in the earlier site-stats split, and saves one CSV per site into `vrm_alarm_logs_split_by_site/`.

The notebook accepts these possible source filenames:

- `site vrm_alarm_logs_clean.csv`
- `site_vrm_alarm_logs_clean.csv`
- `vrm_alarm_logs_clean.csv`


In [ ]:
from pathlib import Path

import pandas as pd

INPUT_CSV = PROJECT_DIR / "data" / "raw" / "vrm_alarm_logs_clean.csv"
OUTPUT_DIR = PROJECT_DIR / "data" / "processed" / "vrm_alarm_logs_split_by_site"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def find_project_root():
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "config" / "sites.csv").exists():
            return candidate
    raise FileNotFoundError(
        "Copy config/sites.example.csv to config/sites.csv before running."
    )


PROJECT_DIR = find_project_root()
SITES_CONFIG = PROJECT_DIR / "config" / "sites.csv"
sites_config = pd.read_csv(SITES_CONFIG)
sites_config["site_id"] = pd.to_numeric(sites_config["site_id"], errors="raise").astype(int)
SELECTED_SITES = dict(zip(sites_config["site_id"], sites_config["site_name"]))

assert INPUT_CSV.exists(), f"Input file not found: {INPUT_CSV}"
print(f"Using input file: {INPUT_CSV.resolve()}")
print(f"Writing output CSVs to: {OUTPUT_DIR.resolve()}")

In [ ]:
alarm_logs = pd.read_csv(INPUT_CSV)
alarm_logs.columns = alarm_logs.columns.str.strip()

site_column = "site_id" if "site_id" in alarm_logs.columns else "idSite" if "idSite" in alarm_logs.columns else None
if site_column is None:
    raise ValueError("Expected either 'site_id' or 'idSite' column in alarm logs")

alarm_logs[site_column] = pd.to_numeric(alarm_logs[site_column], errors="coerce").astype("Int64")

# Parse datetime columns where present, preserving UTC-aware values.
for datetime_column in ["alarm_started_at", "alarm_cleared_at"]:
    if datetime_column in alarm_logs.columns:
        alarm_logs[datetime_column] = pd.to_datetime(alarm_logs[datetime_column], errors="coerce", utc=True)

selected_site_ids = list(SELECTED_SITES)
selected_alarm_logs = alarm_logs[alarm_logs[site_column].isin(selected_site_ids)].copy()

print(f"Total alarm rows loaded: {len(alarm_logs):,}")
print(f"Rows for selected sites: {len(selected_alarm_logs):,}")
print(f"Selected sites found: {selected_alarm_logs[site_column].nunique()} / {len(selected_site_ids)}")

selected_alarm_logs.head()

In [ ]:
summary_rows = []
site_alarm_frames = {}

for site_id in selected_site_ids:
    site_df = selected_alarm_logs[selected_alarm_logs[site_column] == site_id].copy()

    if "alarm_started_at" in site_df.columns:
        site_df = site_df.sort_values("alarm_started_at", ascending=True, na_position="last")

    output_path = OUTPUT_DIR / f"vrm_alarm_logs_{site_id}.csv"
    site_df.to_csv(output_path, index=False)

    globals()[f"vrm_alarm_logs_{site_id}"] = site_df
    site_alarm_frames[site_id] = site_df

    actual_site_name = None
    if "site_name" in site_df.columns and site_df["site_name"].notna().any():
        actual_site_name = site_df["site_name"].dropna().iloc[0]

    summary_rows.append({
        "site_id": site_id,
        "expected_site_name": SELECTED_SITES[site_id],
        "actual_site_name": actual_site_name,
        "rows": len(site_df),
        "active_alarm_rows": int(site_df["isActive"].sum()) if "isActive" in site_df.columns and len(site_df) else 0,
        "unique_alarm_types": site_df["alarm_type"].nunique(dropna=True) if "alarm_type" in site_df.columns else None,
        "unique_classifications": site_df["classification"].nunique(dropna=True) if "classification" in site_df.columns else None,
        "output_csv": str(output_path),
    })

split_summary = pd.DataFrame(summary_rows)
split_summary.to_csv(OUTPUT_DIR / "vrm_alarm_logs_split_summary.csv", index=False)
split_summary

In [ ]:
# Quick check that every requested site produced a file.
created_files = sorted(OUTPUT_DIR.glob("vrm_alarm_logs_*.csv"))
print(f"Created {len(created_files) - 1 if any(path.name == 'vrm_alarm_logs_split_summary.csv' for path in created_files) else len(created_files)} per-site CSV files")
print(f"Summary file: {OUTPUT_DIR / 'vrm_alarm_logs_split_summary.csv'}")

split_summary.sort_values("site_id")

## Alarm source date coverage

In [ ]:
alarm_range_summary = pd.DataFrame([{
    "alarm_rows": len(alarm_logs),
    "valid_alarm_start_rows": int(alarm_logs["alarm_started_at"].notna().sum()),
    "alarm_range_start": alarm_logs["alarm_started_at"].min(),
    "alarm_range_end": alarm_logs["alarm_started_at"].max(),
}])
alarm_range_summary.to_csv(
    OUTPUT_DIR / "vrm_alarm_logs_source_range_summary.csv", index=False
)
alarm_range_summary
